# Huấn luyện Mô hình Phát hiện Rung tâm nhĩ (AFib)
Sử dụng dữ liệu đặc trưng HRV trích xuất từ MIT-BIH AF Database.

Quy trình:
1. **Tải dữ liệu**: Đọc file `mitbih_af_features.csv`
2. **Trực quan hóa (EDA)**: So sánh các đặc trưng giữa Normal và AFib.
3. **Tiền xử lý**: Áp dụng Z-score Normalization để chống Domain Shift khi triển khai lên PPG.
4. **Huấn luyện**: Sử dụng thuật toán Random Forest Classifier.
5. **Đánh giá**: In ra Confusion Matrix và độ chính xác.
6. **Xuất mô hình**: Lưu `scaler.pkl` và `rf_afib_model.pkl`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

# Thiết lập thư mục
os.makedirs('../../models', exist_ok=True)
sns.set_theme(style="whitegrid")

### 1. Tải và kiểm tra dữ liệu

In [ ]:
df = pd.read_csv('../../data/features/mitbih_af_features.csv')
print(f"Tổng số mẫu: {len(df)}")
print("\nPhân phối nhãn:")
print(df['label'].value_counts())

df.head()

### 2. Trực quan hóa Dữ liệu (EDA)
Chúng ta sẽ vẽ biểu đồ RMSSD (Root Mean Square of Successive Differences). 
Về mặt y khoa, người bị AFib nhịp tim rất hỗn loạn, dẫn tới mức độ chênh lệch giữa các nhịp (RMSSD) cao bất thường.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='label', y='rmssd', data=df, showfliers=False, palette='Set2')
plt.title('Phân phối RMSSD: Bình thường (Normal) vs Rung tâm nhĩ (AFib)')
plt.ylabel('RMSSD (ms)')
plt.xlabel('Tình trạng')
plt.show()

### 3. Tiền xử lý & Z-score Normalization
Chuẩn hóa dữ liệu bằng `StandardScaler`. Khi đưa lên cảm biến quang nhịp tim (MAX30102), ta cũng sẽ phải dùng hàm tỷ lệ này để chuẩn hóa nhằm tránh lỗi *Domain Shift* (máy xịn vs máy dởm).

In [ ]:
# Tách features và labels
X = df.drop(columns=['label', 'record'])
y = df['label']

# Chia tập Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Khởi tạo StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Lưu Scaler để dùng trên server AI-Service
joblib.dump(scaler, '../../models/scaler.pkl')
print("Đã lưu StandardScaler vào models/scaler.pkl")

### 4. Huấn luyện Mô hình (Random Forest)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10,
    random_state=42,
    class_weight='balanced' # Xử lý mất cân bằng dữ liệu
)

rf_model.fit(X_train_scaled, y_train)
print("Huấn luyện thành công!")

### 5. Đánh giá Mô hình

In [ ]:
y_pred = rf_model.predict(X_test_scaled)

print("=== BÁO CÁO PHÂN LOẠI ===")
print(classification_report(y_test, y_pred))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=rf_model.classes_, yticklabels=rf_model.classes_)
plt.title('Ma trận nhầm lẫn (Confusion Matrix)')
plt.ylabel('Thực tế (True Label)')
plt.xlabel('Dự đoán (Predicted Label)')
plt.show()

### 6. Xuất File Mô hình
Lưu thuật toán đã học thành file nhị phân `.pkl`

In [ ]:
joblib.dump(rf_model, '../../models/random_forest_afib.pkl')
print("Đã lưu Mô hình Random Forest vào models/random_forest_afib.pkl")
print("Xong quy trình Machine Learning!")